Step 1: Loading a dataset, and cleaning them

In [6]:
import pandas as pd

# Loading a dataset
data = pd.read_csv("synthetic_heart_disease_dataset.csv",keep_default_na=False)

# important : Gender column removed because its not important feature to model, it can reduce accuracy
data = data.drop("Gender",axis=1)

# spliting the data into two (features/input and target/ouput)
x = data.iloc[:, :19]
y = data.iloc[:, 19]

# just cross verifying
# print(x.head)
# print(y.head)
# print(x.shape)
# print(y.shape)

Step 2: Converting text values to numerical value using LabelEncoder

In [7]:
from sklearn.preprocessing import LabelEncoder
encoders={}
text_values_columns = ["Smoking","Alcohol_Intake","Physical_Activity","Diet","Stress_Level"]

for col in text_values_columns:
    le = LabelEncoder() 
    x[col] = le.fit_transform(x[col])
    encoders[col] = le

# just cross verifying
# print(x.head)

Step 3: Spliting the dataset into two parts (train and test dataset)

In [8]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)

# just cross verifying
# print(len(x_train),len(y_train)) #train
# print(len(x_test),len(y_test)) #test

Step 4: Normalizing the values using StandardScaler

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_scaled = scaler.fit_transform(x_train) # learns and applies
x_test = scaler.transform(x_test) # it make changes from what it learn

Step 5: Training the model

In [10]:
from sklearn.metrics import accuracy_score, classification_report,confusion_matrix
from sklearn.linear_model import LogisticRegression
import joblib

# The model use LogisticRegression Algo to train the data 
model=LogisticRegression(max_iter=1000) 

# training the data
model.fit(x_scaled,y_train)

#storing the trained model in a pickle file
joblib.dump({
    "model":model,
    "scaler":scaler,
    "encoders":encoders
    },
    "heart_disease_dataset.pkl"
)

['heart_disease_dataset.pkl']

Step 6:Checking the accuracy of the model

In [11]:
# predicting the output on untrained data
y_pred = model.predict(x_test)

# checking model accuracy
print(accuracy_score(y_test,y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

0.9263
              precision    recall  f1-score   support

           0       0.93      0.93      0.93      5342
           1       0.92      0.92      0.92      4658

    accuracy                           0.93     10000
   macro avg       0.93      0.93      0.93     10000
weighted avg       0.93      0.93      0.93     10000

[[4979  363]
 [ 374 4284]]


Step 7: Function to predict the output for new input

In [22]:
import joblib

def predict(new_data):
    saved = joblib.load("heart_disease_dataset.pkl")

    model = saved["model"]
    scaler = saved["scaler"]
    encoders = saved["encoders"]

    new_data = new_data.drop(["Gender","output"],axis=1)

    for col in text_values_columns:
        new_data[col] = encoders[col].transform(new_data[col])

    new_data_scaled = scaler.transform(new_data)

    prediction = model.predict(new_data_scaled)

    return prediction

Step 8: Predicting the output by giving inputs

In [ ]:
from time import sleep
from IPython.display import clear_output
import json


# importing data from json file
with open("unseenedData.json",'r') as new_data_file:
  new_data = json.load(new_data_file)

# new data converted to DataFrame
new_data = pd.DataFrame(new_data)


# counts only if the model predicts correct output
count_of_correct_prediction = 0
N = len(new_data) # lenght of the dataset

for i in range (0,N):
  # sleep(0.5) # 0.5 second delay
  prediction = predict(new_data.iloc[[i]])
  
  if(prediction[0]==new_data.iloc[i]["output"]):
    print(f"{i} Row Prediction ✔️: ",prediction)
    count_of_correct_prediction+=1
  else:
    print(f"{i} Prediction Row ❌: ",prediction)
  clear_output() # clears terminal outputs

  

print(f"Total Data : {N}")
print(f"Right Prediction ✔️: {count_of_correct_prediction}")
print(f"Wrong Prediction ❌: {N-count_of_correct_prediction}")

Total Data : 124
Right Prediction ✔️: 123
Wrong Prediction ❌: 1
